#LSTM (LONG SHORT TERM MEMORY)

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Once your Drive is mounted, you can load your data by specifying the path to the file in your Drive. Replace `'path/to/your/file.csv'` with the actual path to your file.

In [4]:
import pandas as pd

# Replace 'path/to/your/file.csv' with the actual path to your file in Google Drive
file_path = '/content/drive/MyDrive/IMDB Dataset.csv'

try:
    df = pd.read_csv(file_path)
    display(df.head())
except FileNotFoundError:
    print(f"Error: The file was not found at {file_path}")
except Exception as e:
    print(f"An error occurred: {e}")

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [5]:
df.shape


(50000, 2)

In [6]:
df["sentiment"].value_counts()

,count
sentiment,
positive,25000
negative,25000


#One Hot Encoding

In [7]:
df.replace({"sentiment": {"positive":1, "negative": 0}}, inplace=True)
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,1
1,A wonderful little production. <br /><br />The...,1
2,I thought this was a wonderful way to spend ti...,1
3,Basically there's a family where a little boy ...,0
4,"Petter Mattei's ""Love in the Time of Money"" is...",1


In [8]:
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Embedding, LSTM
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [9]:
train_df, test_df = train_test_split(df, test_size = 0.2, random_state=42)

In [10]:
train_df.shape

(40000, 2)

In [11]:
test_df.shape

(10000, 2)

In [12]:
tokenizer = Tokenizer(num_words = 5000)
tokenizer.fit_on_texts(train_df["review"])

In [13]:
X_train = pad_sequences(tokenizer.texts_to_sequences(train_df["review"]), maxlen=200)
X_test = pad_sequences(tokenizer.texts_to_sequences(test_df["review"]), maxlen=200)

In [14]:
X_train

array([[1935,    1, 1200, ...,  205,  351, 3856],
       [   3, 1651,  595, ...,   89,  103,    9],
       [   0,    0,    0, ...,    2,  710,   62],
       ...,
       [   0,    0,    0, ..., 1641,    2,  603],
       [   0,    0,    0, ...,  245,  103,  125],
       [   0,    0,    0, ...,   70,   73, 2062]], dtype=int32)

In [15]:
X_test

array([[   0,    0,    0, ...,  995,  719,  155],
       [  12,  162,   59, ...,  380,    7,    7],
       [   0,    0,    0, ...,   50, 1088,   96],
       ...,
       [   0,    0,    0, ...,  125,  200, 3241],
       [   0,    0,    0, ..., 1066,    1, 2305],
       [   0,    0,    0, ...,    1,  332,   27]], dtype=int32)

In [16]:
Y_train = train_df["sentiment"]
Y_test = test_df["sentiment"]

In [17]:
model = Sequential()
model.add(Embedding(input_dim=5000, output_dim=128, input_length=200))
model.add(LSTM(units=128, dropout=0.2, recurrent_dropout=0.2))
model.add(Dense(units=1, activation="sigmoid"))

In [18]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [19]:
model.compile(optimizer = "adam", loss="binary_crossentropy", metrics=["accuracy"])

In [20]:
model.fit(X_train, Y_train, batch_size=64, epochs=5, validation_split = 0.2)

Epoch 1/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 214s 409ms/step - accuracy: 0.7203 - loss: 0.5322 - val_accuracy: 0.8094 - val_loss: 0.4180
Epoch 2/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 205s 409ms/step - accuracy: 0.8554 - loss: 0.3433 - val_accuracy: 0.8665 - val_loss: 0.3287
Epoch 3/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 264s 413ms/step - accuracy: 0.8862 - loss: 0.2835 - val_accuracy: 0.8568 - val_loss: 0.3304
Epoch 4/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 204s 408ms/step - accuracy: 0.8904 - loss: 0.2711 - val_accuracy: 0.8677 - val_loss: 0.3246
Epoch 5/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 204s 407ms/step - accuracy: 0.8751 - loss: 0.2938 - val_accuracy: 0.8695 - val_loss: 0.3436


In [21]:
loss, accuracy = model.evaluate(X_test, Y_test)

313/313 ━━━━━━━━━━━━━━━━━━━━ 42s 133ms/step - accuracy: 0.8777 - loss: 0.3246


In [22]:
print(loss)

0.3264836370944977


In [23]:
print(accuracy)

0.8776000142097473


#Building PRedictive System

In [24]:
def predictive_system(review):
  sequences = tokenizer.texts_to_sequences([review])
  padded_sequences = pad_sequences(sequences, maxlen=200)
  prediction = model.predict(padded_sequences)
  sentiment = "positive" if prediction[0][0] > 0.5 else "negative"
  return sentiment

In [25]:
predictive_system("This movie is fantastic!")

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 879ms/step


'positive'

In [26]:
model.save("model.h5")

In [27]:
    import joblib
joblib.dump(tokenizer, "tokenizer.pkl")

['tokenizer.pkl']